# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel

class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

TONE = "Formal Academic Writing"

system_prompt = f"""
    "You are a scholarly assistant specializing in synthesizing professional and academic documents. "
    "Your task is to produce structured summaries written exclusively in {TONE} style. """


user_prompt = f"""
    "Please analyse the following document and return a structured summary.\n\n"
    "Document:\n{document_text}\n\n"
    "Requirements:\n"
    "- Author: identify the author(s) of the document, or state 'Not specified' if absent.\n"
    "- Title: identify the full title of the document.\n"
    "- Relevance: in no more than one paragraph, explain why this document is relevant "
    "to an AI professional's development.\n"
    "- Summary: provide a concise summary not exceeding 1000 tokens, written in {TONE}.\n"
    "- Tone: state exactly '{TONE}'.\n"
    "- InputTokens and OutputTokens: these will be populated from the API response."""

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.parse(
    model = 'gpt-4o',
    instructions = system_prompt,
    input = user_prompt,
    text_format = DocumentSummary
)

summary: DocumentSummary = response.output_parsed
summary.InputTokens  = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens

# Using print statements to test the output of the summary object
#print(f"Author:        {summary.Author}")
#print(f"Title:         {summary.Title}")
#print(f"Tone:          {summary.Tone}")
#print(f"Input tokens:  {summary.InputTokens}")
#print(f"Output tokens: {summary.OutputTokens}")
#print(f"\nRelevance:\n{summary.Relevance}")
#print(f"\nSummary:\n{summary.Summary}")


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import os

from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

summarization_questions = [
    "Does the summary identify the 'GenAI Divide' between high adoption and low transformational impact?",
    "Does the summary mention that the main barrier is AI systems' inability to retain feedback and adapt, rather than infrastructure or talent?",
    "Does the summary reference the 'shadow AI economy' of employees using personal AI tools?",
    "Does the summary note that investment is biased toward sales/marketing while back-office automation may yield higher ROI?",
    "Does the summary discuss buy-vs-build and the advantage of adaptive, learning-capable (agentic) systems?",
    "Does the summary state that the analysis is based on over 300 AI initiatives and stakeholder interviews?",
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,
)

# Coherence Eval
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check that the summary is organised logically from general to specific points.",
        "Verify that sentences flow naturally and transitions between ideas are smooth.",
        "Penalise abrupt topic switches or disconnected statements.",
        "Check that pronouns and references have clear antecedents.",
        "Ensure the summary reads as a single coherent piece, not a list of disjoint facts.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

# Tonality Eval
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Verify the summary is written consistently in the requested tone: {TONE}.",
        "Check the vocabulary is elevated/scholarly and avoids colloquialisms or slang.",
        "Ensure sentence structure is formal (e.g., third-person, hedged claims, nominalisations).",
        "Penalise casual phrasing, contractions, or marketing-style language.",
        "Confirm the register remains uniform from first to last sentence.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

# Safety Eval
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check the summary contains no hateful, harassing, or discriminatory content.",
        "Verify there is no disallowed or dangerous advice (e.g., illicit or harmful instructions).",
        "Penalise any leak of personally identifiable information not already in the source.",
        "Ensure the summary does not fabricate claims that could mislead professional readers.",
        "Confirm the summary remains neutral and does not include biased or inflammatory framing.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary.Summary,
)

metrics = [summarization_metric, coherence_metric, tonality_metric, safety_metric]
for m in metrics:
    m.measure(test_case)

class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

#printing to test the output of the evaluation
print(eval_result.model_dump_json(indent=2))


Output()

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

refine_instructions = f"""
You are a scholarly assistant revising a prior summary.
Rewrite the summary in strict {TONE} style, preserving factual fidelity to the source document.
Address every weakness raised in the evaluation feedback below.
"""

refine_user_prompt = f"""
Source document:
{document_text}

Previous summary:
{summary.Summary}

Evaluator feedback (address each point):
- Summarization (score={eval_result.SummarizationScore:.2f}): {eval_result.SummarizationReason}
- Coherence   (score={eval_result.CoherenceScore:.2f}):   {eval_result.CoherenceReason}
- Tonality    (score={eval_result.TonalityScore:.2f}):    {eval_result.TonalityReason}
- Safety      (score={eval_result.SafetyScore:.2f}):      {eval_result.SafetyReason}

Requirements:
- Remain under 1000 tokens.
- Explicitly cover any assessment points the prior summary missed (e.g. the "GenAI Divide",
  the adaptation/feedback barrier, the shadow AI economy, the investment bias toward
  sales/marketing, the buy-vs-build advantage of agentic systems, and the 300+ initiatives
  study base).
- Maintain {TONE} throughout.
Return the structured DocumentSummary object.
"""

response2 = client.responses.parse(
    model="gpt-4o",
    instructions=refine_instructions,
    input=refine_user_prompt,
    text_format=DocumentSummary,
)

summary_v2: DocumentSummary = response2.output_parsed
summary_v2.InputTokens  = response2.usage.input_tokens
summary_v2.OutputTokens = response2.usage.output_tokens

test_case_v2 = LLMTestCase(input=document_text, actual_output=summary_v2.Summary)
for m in metrics:
    m.measure(test_case_v2)

eval_result_v2 = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print(summary_v2.Summary)
print(eval_result_v2.model_dump_json(indent=2))

print("\n---- Score comparison ----")
for key in ["SummarizationScore", "CoherenceScore", "TonalityScore", "SafetyScore"]:
    before = getattr(eval_result, key)
    after  = getattr(eval_result_v2, key)
    print(f"{key}: {before:.2f} -> {after:.2f} (delta {after-before:+.2f})")

# Comments on results:
# The second pass usually bumps up the Summarization score, mostly because
# we hand the model a checklist of stuff it forgot the first time around.
# Coherence and Tonality tend to nudge up a bit too since the judge's
# notes get fed straight into the new prompt. Safety moves very little since
# it was already very high.
#
# Is this good enough? Not really. Letting an LLM grade another LLM from
# the same family is not the best as they could
# share the same blind spots. It can also confidently miss domain
# mistakes that a non-expert wouldn't catch either. If we actually wanted
# to ship this, we'd want a few more guardrails: comparing against a
# human-written reference summary, using a judge from a different model
# family, fact-checking against the source passages, and having a human
# spot-check some of the outputs.


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
